# NS-SHAFT Colab pipeline validation

This notebook is intentionally simulator-only. It never uploads/runs the game executable and never creates a Windows input backend. The optional 768-step PPO cell validates checkpoint/save/resume/video plumbing only; BC, DAgger, DQfD and long training remain out of scope.

## Private repository / manual upload setup

The setup cell defaults to manual upload and does not require GitHub credentials. Download or create a ZIP of the repository on your computer, upload that ZIP when prompted, and keep the repository structure intact. The ZIP may contain a top-level folder; the cell locates `pyproject.toml` automatically. If the project was already extracted under `/content`, it is reused without prompting.

In [ ]:
import os
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'
print('Headless SDL configured')

In [ ]:
import os
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

# Default for a private repository: upload one repository ZIP manually.
# If you already extracted the repository in Colab, set this to that folder.
MANUAL_PROJECT_PATH = ''  # Example: '/content/stairkid-rl-main'
CONTENT_ROOT = Path('/content')

def project_candidates(root):
    root = Path(root)
    if not root.exists():
        return []
    return sorted({
        marker.parent.resolve()
        for marker in root.rglob('pyproject.toml')
        if (marker.parent / 'src' / 'stair_agent').is_dir()
    })

def archive_contains_project(archive):
    try:
        with zipfile.ZipFile(archive) as bundle:
            names = {name.replace('\\\\', '/') for name in bundle.namelist()}
        markers = [name for name in names if name.endswith('pyproject.toml')]
        return any(
            f"{marker[:-len('pyproject.toml')]}src/stair_agent/" in names
            or any(name.startswith(f"{marker[:-len('pyproject.toml')]}src/stair_agent/") for name in names)
            for marker in markers
        )
    except zipfile.BadZipFile:
        return False

def safe_extract(archive):
    destination = Path(tempfile.mkdtemp(prefix='stairkid-upload-', dir=CONTENT_ROOT))
    destination_root = destination.resolve()
    with zipfile.ZipFile(archive) as bundle:
        for item in bundle.infolist():
            target = (destination / item.filename).resolve()
            if target != destination_root and destination_root not in target.parents:
                raise RuntimeError(f'Unsafe path in ZIP: {item.filename}')
        bundle.extractall(destination)
    return destination

if MANUAL_PROJECT_PATH:
    candidates = project_candidates(Path(MANUAL_PROJECT_PATH).expanduser())
else:
    candidates = project_candidates(CONTENT_ROOT)

if not candidates:
    archives = [path for path in CONTENT_ROOT.glob('*.zip') if archive_contains_project(path)]
    if not archives:
        from google.colab import files
        print('Upload the repository ZIP (not a checkpoint ZIP).')
        uploaded = files.upload()
        archives = [CONTENT_ROOT / name for name in uploaded if archive_contains_project(CONTENT_ROOT / name)]
    if len(archives) != 1:
        raise RuntimeError(f'Expected exactly one repository ZIP, found {len(archives)}: {archives}')
    candidates = project_candidates(safe_extract(archives[0]))

if len(candidates) != 1:
    raise RuntimeError(f'Expected exactly one NS-SHAFT project, found {len(candidates)}: {candidates}')

REPO_DIR = candidates[0]
os.chdir(REPO_DIR)
print(f'Project located at: {REPO_DIR}')
assert (REPO_DIR / 'pyproject.toml').is_file()
python_version = sys.version_info[:2]
print('Python runtime:', sys.version)
if not ((3, 11) <= python_version < (3, 13)):
    raise RuntimeError(f'Unsupported Python {python_version}; expected Python 3.11 or 3.12')

from importlib.metadata import version
setuptools_version = version('setuptools')
setuptools_major = int(setuptools_version.split('.', 1)[0])
print('Setuptools:', setuptools_version)
if setuptools_major < 65:
    raise RuntimeError('setuptools>=65 is required before installing this project')

# Do not use pip -q here: Colab must show the actual resolver/build error if installation fails.
# Colab already provides setuptools; reusing it avoids a redundant isolated build download.
install_command = [
    sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
    '--no-build-isolation', '-e', '.[test,rl]',
    'tensorboard', 'imageio[ffmpeg]'
]
print('Installing project and dependencies from:', REPO_DIR)
install_result = subprocess.run(install_command, cwd=REPO_DIR)
if install_result.returncode != 0:
    raise RuntimeError(
        f'pip install failed with exit code {install_result.returncode}. '
        'Read the complete pip output immediately above this message.'
    )

import gymnasium
import pymunk
import stable_baselines3
import stair_agent

print('Installed ai-stair-agent:', version('ai-stair-agent'))
print('Gymnasium:', gymnasium.__version__)
print('Pymunk:', version('pymunk'))
print('Stable-Baselines3:', stable_baselines3.__version__)
print('Installation and import checks passed; Windows-only input packages were skipped.')

## Optional Google Drive mount

Run this only when persistent artifacts are needed. Use experiment IDs; do not overwrite a sole `latest` checkpoint.

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/ns-shaft-runs') if USE_DRIVE else Path('/content/ns-shaft-runs')
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT

## Tests, Gymnasium contract and headless smoke

In [ ]:
!python -m pytest -q
!python scripts/check_simulator.py --steps 10000 --baseline-steps 1000 --seed 0

In [ ]:
from gymnasium.utils.env_checker import check_env
from stair_agent.envs.shaft_env import ShaftEnv
env = ShaftEnv()
check_env(env, skip_render_check=True)
env.close()
print('check_env passed')

## Sync/async vector benchmark (1, 4, 8, 16 envs)

The recommendation is throughput-based and must still be checked against Colab RAM/CPU utilization.

In [ ]:
import time
import numpy as np
from gymnasium.vector import AsyncVectorEnv, SyncVectorEnv
from stair_agent.envs.shaft_env import ShaftEnv

def make_env():
    return ShaftEnv()

def benchmark(vector_cls, count, vector_steps=2000):
    vector = vector_cls([make_env for _ in range(count)])
    try:
        vector.reset(seed=list(range(count)))
        started = time.perf_counter()
        for _ in range(vector_steps):
            vector.step(np.zeros(count, dtype=np.int64))
        elapsed = time.perf_counter() - started
        return count * vector_steps / elapsed
    finally:
        vector.close()

results = []
for count in (1, 4, 8, 16):
    for name, vector_cls in (('sync', SyncVectorEnv), ('async', AsyncVectorEnv)):
        rate = benchmark(vector_cls, count)
        results.append({'mode': name, 'envs': count, 'steps_per_second': rate})
        print(name, count, f'{rate:.0f} steps/s')
recommended = max(results, key=lambda item: item['steps_per_second'])
print('Throughput recommendation:', recommended)

## Versioned artifact paths, TensorBoard and bounded pipeline validation

In [ ]:
from datetime import datetime, timezone
import json
EXPERIMENT_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ_sim_v0_probe')
RUN_DIR = ARTIFACT_ROOT / EXPERIMENT_ID
for name in ('tensorboard', 'checkpoints', 'videos'):
    (RUN_DIR / name).mkdir(parents=True, exist_ok=False)
(RUN_DIR / 'config.json').write_text(json.dumps({
    'experiment_id': EXPERIMENT_ID,
    'observation_schema': 'stair-observation-v3-268',
    'purpose': 'bounded Colab runtime/checkpoint/resume/video validation'
}, indent=2) + '\n')
print(RUN_DIR)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $RUN_DIR/tensorboard

In [ ]:
RUN_COLAB_PIPELINE_VALIDATION = False  # Set True only after tests/benchmark cells pass.
if not RUN_COLAB_PIPELINE_VALIDATION:
    print('Validation disabled: set RUN_COLAB_PIPELINE_VALIDATION=True to run the frozen 768-step smoke.')
else:
    import imageio.v3 as iio
    import numpy as np
    import torch
    from stable_baselines3 import PPO
    from stair_agent.envs.shaft_env import ShaftEnv, ShaftEnvConfig

    train_env = ShaftEnv(config=ShaftEnvConfig(max_episode_steps=120))
    model = PPO(
        'MlpPolicy', train_env, n_steps=128, batch_size=64, n_epochs=1,
        policy_kwargs={'net_arch': [64, 64]}, seed=51001,
        device='auto', tensorboard_log=str(RUN_DIR / 'tensorboard'), verbose=0,
    )
    model.learn(total_timesteps=512, progress_bar=False)
    initial_checkpoint = RUN_DIR / 'checkpoints' / 'ppo_000512'
    model.save(initial_checkpoint)
    loaded = PPO.load(str(initial_checkpoint) + '.zip', env=train_env, device='auto')
    before_resume = loaded.num_timesteps
    loaded.learn(total_timesteps=256, reset_num_timesteps=False, progress_bar=False)
    resumed_checkpoint = RUN_DIR / 'checkpoints' / 'ppo_000768_resumed'
    loaded.save(resumed_checkpoint)
    assert loaded.num_timesteps >= before_resume + 256
    train_env.close()

    video_env = ShaftEnv(
        config=ShaftEnvConfig(max_episode_steps=120), render_mode='rgb_array'
    )
    observation, _ = video_env.reset(seed=52001)
    frames = [video_env.render()]
    for _ in range(120):
        action, _ = loaded.predict(observation, deterministic=True)
        observation, _, terminated, truncated, _ = video_env.step(int(action))
        frames.append(video_env.render())
        if terminated or truncated:
            break
    video_env.close()
    video_path = RUN_DIR / 'videos' / 'ppo_resumed_eval.mp4'
    iio.imwrite(video_path, np.asarray(frames), fps=8, codec='libx264')

    gate = {
        'initial_checkpoint': (str(initial_checkpoint) + '.zip'),
        'resumed_checkpoint': (str(resumed_checkpoint) + '.zip'),
        'video': str(video_path),
        'video_frames': len(frames),
        'torch_device': str(loaded.device),
        'cuda_available': torch.cuda.is_available(),
        'pipeline_pass': all([
            Path(str(initial_checkpoint) + '.zip').is_file(),
            Path(str(resumed_checkpoint) + '.zip').is_file(),
            video_path.is_file(), len(frames) >= 2,
        ]),
    }
    (RUN_DIR / 'colab_pipeline_gate.json').write_text(
        json.dumps(gate, indent=2) + '\n', encoding='utf-8'
    )
    print(json.dumps(gate, indent=2))
    assert gate['pipeline_pass']